### start with import first

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import sympy as sp

#use pd.read_csv() to obtain 2 dataframes
strandInfo = pd.read_csv('../data/Metadata_n1932.csv')
regionInfo = pd.read_csv('../data/region.csv')
# strandInfo

### not sure what to do yet, plans in google doc; read fasta file first

In [ ]:
#here, the fasta file is read in as a dictionary "seq"
seq = dict()

with open('../data/n1932_29903.fas') as f:
    lines = f.readlines()
for i in range(0, len(lines)):
    s = lines[i].strip()
    if s[0] == '>':
        key = s[1:]
    else:
        seq[key] = s
dict(list(seq.items())[0:3])

In [ ]:
#getting all the values from the fasta dictionary and appending them into a matrix
matVal = []
for i in dict(list(seq.items())).values():
    matVal.append(list(i))



#len(matVal)

In [ ]:
n=[]
a = 0
gap = 0
t = 0
g = 0
c = 0
for i in matVal:
    n.append(i[0])

pd.Series(n).value_counts()

In [ ]:
#sorting out the list that contains all the id of the sequences
matKey = []
for i in dict(list(seq.items())).keys():
    matKey.append(i)
    
#matKey[:2]

In [ ]:
#isolate the reference
refIndex = 0
for i in range(0,len(matKey)):
    if matKey[i].__eq__('MN908947'):
        refIndex = i
        
refSeqVal = matVal[refIndex]

refSeqVal

In [ ]:
#catagorize the matrix since the dimensions are too large for computations

#get an array of country info
arrCountry = strandInfo.loc[:,'Country'].tolist()
arrID = strandInfo.loc[:,'ID'].tolist()
arrTime = strandInfo.loc[:,'Collection.Date'].tolist()

#filter the data into US and China
arrID_US = []
arrID_China = []
arrID_Taiwan = []
arrID_Italy = []
arrID_Japan = []
arrID_SK = []
arrID_UK = []
arrID_Belgium = []
arrID_France = []

    
#looping through the arr, getting rid of non-US & non-China data
for i in reversed(range(len(arrCountry))):
    if arrCountry[i] == arrCountry[3]: #i really don't know why i have to do this... but i can't get the string 'United States' other wise
        arrID_US.append(arrID[i])
    elif arrCountry[i] == arrCountry[94]:
        arrID_China.append(arrID[i])
    elif arrCountry[i] == arrCountry[56]:
        arrID_Taiwan.append(arrID[i])
    elif arrCountry[i] == arrCountry[0]:
        arrID_Italy.append(arrID[i])
    elif arrCountry[i] == arrCountry[66]:
        arrID_Japan.append(arrID[i])
    elif arrCountry[i] == arrCountry[1775]:
        arrID_SK.append(arrID[i])
    elif arrCountry[i] == arrCountry[173]:
        arrID_UK.append(arrID[i])
    elif arrCountry[i] == arrCountry[194]:
        arrID_Belgium.append(arrID[i])
    elif arrCountry[i] == arrCountry[285]:
        arrID_France.append(arrID[i])

#print(arrID_US)
#print(len(arrID_China))
#print(arrID_Taiwan)

In [ ]:
#one extra step: remove reference from China, the it wouldn't be recounted
arrID_China.remove('MN908947')

In [ ]:
#function for finding the seq val of each country
def finding_seq(arrID):
    seq = []
    for i in range(0,len(arrID)):
        for ii in range(0,len(matKey)):
            if arrID[i]==matKey[ii]:
                seq.append(matVal[ii])
    return seq

In [ ]:
seqUS = finding_seq(arrID_US)
seqChina = finding_seq(arrID_China)
seqTaiwan = finding_seq(arrID_Taiwan)
seqItaly = finding_seq(arrID_Italy)
seqJapan = finding_seq(arrID_Japan)
seqSK = finding_seq(arrID_SK)
seqUK = finding_seq(arrID_UK)
seqBelgium = finding_seq(arrID_Belgium)
seqFrance = finding_seq(arrID_France)

In [ ]:
#the "similarity rate " of the sequences of sampled virus strains in diff regions

def cal_rate(seq):
    arrCount = []
    count = 0
    for i in seq:
        for ii in range(0,len(i)):
            if i[ii] == refSeqVal[ii]:
                count+=1
        arrCount.append(count)
        count = 0
    sumCount = sum(arrCount)
    return sumCount/(29903*len(arrCount))

In [ ]:
rateTaiwan = cal_rate(seqTaiwan)
rateUS = cal_rate(seqUS)
rateChina = cal_rate(seqChina)
rateItaly = cal_rate(seqItaly)
rateJapan = cal_rate(seqJapan)
rateSK = cal_rate(seqSK)
rateUK = cal_rate(seqUK)
rateBelgium = cal_rate(seqBelgium)
rateFrance = cal_rate(seqFrance)

In [ ]:
#make it into a graph - thinking of bar graph right now...
plt.bar(['US','China','Taiwan','Italy','Japan','South Korea','UK','Belgium','France'], [rateUS,rateChina,rateTaiwan,rateItaly,rateJapan,rateSK,rateUK,rateBelgium,rateFrance], color ='maroon',
        width = 0.4)
plt.ylim(0.996,1)

plt.xlabel("Different countries")
plt.ylabel("Similarity rate of strands to the reference")
plt.title("The Similarity Rate of RNA Strands to Reference in Different Countries")
plt.show()

In [ ]:
# #i want to make a relation between mutation of rna and time, no plans right now...
# ##start w conversion of string to date to value(for calculations)

# arrDateStr = strandInfo.loc[:,'Collection.Date'].tolist()
# # dates = []
# # for i in arrDateStr:
# #     dates.append(datetime.strptime(i, '%y/%m/%d'))
    
# # dates
# arrDateStr


In [ ]:
#organize the data into binary
## IMPORTANT: 0 means no change; 1 = occurence change
matVal_binary = matVal
for i in range(0,len(matVal)):
    for j in range(0,len(matVal[i])):
        if (matVal[i])[j]==refSeqVal[j]:
            matVal_binary[i][j] = 0
        else:
            matVal_binary[i][j] = 1
            


In [ ]:
toRemove = []
count = 0
for i in range(2,len(matVal_binary[0])):
    for ii in range(0,len(matVal_binary)):
        count += matVal_binary[ii][i]
    if count == 0:
        toRemove.append(i)
    count = 0

In [ ]:
for i in range(0,len(matVal_binary)):
    matVal_binary[i].insert(0,arrTime[i])
    matVal_binary[i].insert(0,arrCountry[i])

In [ ]:
matVal == matVal_binary